# R51 — one controlled training question

Does giving **twice the classification BCE weight to anchors assigned to partial cassettes** improve detection while preserving corner quality?

Both arms use exactly the **same 142 training photographs and annotation bytes as the PR137 candidate**, including five supplementary frame photos. Both start fresh from epoch173, use seed46, and run the same 30-epoch recipe. The reference factor1 is checked against the native loss; the candidate factor2 changes only the selected classification terms. This is separate from the earlier box-only and combined box/classification experiments.

In Google Colab select **Runtime → Change runtime type → T4 GPU**, then **Run all**. Four code cells perform source/data checks, CPU correctness checks, a four-batch resource trial, and the two matched runs. Return **`baomai-partial-classification-r51-results.zip`**.

**Publication:** the source revision and driver SHA256 are pinned below. No R51 GPU training result is claimed by this notebook. PR35 remains selected; nothing is automatically promoted.

In [ ]:
# 1. Restore the exact source and verify the complete local Python closure.
import hashlib, json, os, re, subprocess, sys
from pathlib import Path
SOURCE_REVISION = 'da5a67d4bec8171246c56c6b03b34ee20fc519e0'
SCRIPT_SHA256 = '382b51e48013ec82b30eb1a44175546ae30ce716bd75882646338ae8b48b0852'
assert re.fullmatch(r'[0-9a-f]{40}', SOURCE_REVISION), 'Publication incomplete: source revision must be pinned.'
assert re.fullmatch(r'[0-9a-f]{64}', SCRIPT_SHA256), 'Publication incomplete: script SHA256 must be pinned.'
BASE = Path(os.environ.get('BAOMAI_R51_BASE', '/content/baomai-r51-' + SOURCE_REVISION[:12]))
SOURCE, WORK = BASE / 'source', BASE / 'run'
REPOSITORY = os.environ.get('BAOMAI_R51_REPOSITORY', 'https://github.com/Siuuuers/dl.git')
ENV = dict(os.environ, GIT_LFS_SKIP_SMUDGE='1', GIT_TERMINAL_PROMPT='0', PYTHONDONTWRITEBYTECODE='1')
def git(*args, capture=False, text_input=None):
    result = subprocess.run(['git', '-C', str(SOURCE), *args], check=True, env=ENV,
                            input=text_input, text=True, capture_output=capture)
    return result.stdout.strip() if capture else None
FRESH = not (SOURCE / '.git').exists()
if not FRESH:
    assert git('rev-parse', 'HEAD', capture=True) == SOURCE_REVISION, 'Wrong checkout preserved. Choose a fresh BAOMAI_R51_BASE.'
    assert not git('status', '--porcelain', '--untracked-files=all', capture=True), 'Modified checkout preserved. Choose a fresh BAOMAI_R51_BASE.'
else:
    assert not SOURCE.exists() or not any(SOURCE.iterdir()), 'Existing source files preserved. Choose a fresh BAOMAI_R51_BASE.'
    SOURCE.mkdir(parents=True, exist_ok=True)
    git('init', '-q')
    git('remote', 'add', 'origin', REPOSITORY)
    git('fetch', '--depth=1', '--filter=blob:none', 'origin', SOURCE_REVISION)
patterns = ['/scripts/', '/data/manifest.json', '/data/labels/',
            '/data/candidates/front-review-2026-11-26-r37/',
            '/data/reviews/additional-fronts-2026-11-27-r47/',
            '/data/reviews/fronts-2026-11-26-r38/', '/data/reviews/fronts-2026-11-27-r50/',
            '/data/references/sequence-refined-reference-2026-11-18/',
            '/data/references/visible-front-completion-2026-11-14/']
git('sparse-checkout', 'init', '--no-cone')
git('sparse-checkout', 'set', '--no-cone', '--stdin', text_input='\n'.join(patterns) + '\n')
if FRESH: git('checkout', '--detach', SOURCE_REVISION)
SCRIPT = SOURCE / 'scripts/run_partial_classification_r51.py'
assert hashlib.sha256(SCRIPT.read_bytes()).hexdigest() == SCRIPT_SHA256, 'Driver hash differs.'
COMMON = ['--root', str(SOURCE), '--work', str(WORK), '--source-revision', SOURCE_REVISION, '--script-sha256', SCRIPT_SHA256]
def run(mode, *extra):
    subprocess.run([sys.executable, '-B', str(SCRIPT), mode, *COMMON, *extra], check=True, env=ENV)
run('source-check')


## Verify the data before installing the training package

The exact PR137 archive is audited and its candidate labels are copied into both arms. Original PNGs, five original JPEGs, and the epoch173 initializer are restored from immutable sources and checked by SHA256 and image dimensions. No review chart regeneration is needed.

Each arm has **142 training images, 3,999 cassette targets and 13 empty training images**. The native training validation split remains **20 images / 664 targets**. Final checkpoint evaluation separately scores frozen666, historical663, R38 and R50 on those same reused development scenes. Later supplementary changes, including the R48 frame_0001 candidate, are not inserted into this isolated experiment.

In [ ]:
# 2. CPU source/data preflight; preserve incompatible existing work.
ARCHIVE = Path(os.environ.get('BAOMAI_R51_ARCHIVE', str(WORK / 'cache/pr137/baomai-additional-front-pilot-r47-results.zip')))
INPUTS = Path(os.environ.get('BAOMAI_R51_INPUTS', str(WORK / 'cache/assets')))
if os.environ.get('BAOMAI_R51_ARCHIVE'):
    COMMON += ['--archive', str(ARCHIVE)]
COMMON += ['--inputs-dir', str(INPUTS)]
run('prepare', '--download-inputs')
PLAN = json.loads((WORK / 'staging.json').read_text())
print('Both arm inventories:', PLAN['counts'])
print('Each arm budget:', {k: PLAN['budgets']['reference'][k] for k in ('epochs', 'batches', 'images', 'steps')})
RESULT = BASE / 'baomai-partial-classification-r51-results.zip'
COMPLETED = RESULT.exists()
if COMPLETED:
    run('restore-return', '--output', str(RESULT))
    print('Completed return verified; installation and training will be skipped.')


## Check native equivalence and the actual loaders on CPU

The guard uses a small CPU input only to verify loss and gradients. Factor1 must match the native loss and all network gradients bit for bit. Factor2 must double only the selected classification-logit gradients while leaving box, keypoint and sigma gradients unchanged. Both native branches, empty examples, full targets and consumed visibility changes are checked.

The real 142-photo datasets are then built with the pinned training loader. Every source image, label byte and cached coordinate is checked. Production training still uses **imgsz1280 / batch4 / FP32 / AdamW**. The CPU guard performs no optimizer update.

A larger loss weight does not guarantee larger confidence: the native class target is a soft assignment score. This experiment tests its net effect at the existing confidence threshold.

In [ ]:
# 3. Install without replacing the existing torch/torchvision pair, then CPU gates.
import importlib.metadata as metadata
if not COMPLETED:
    constraints = WORK / 'runtime-constraints.txt'
    constraints.write_text('\n'.join(name + '==' + metadata.version(name) for name in ('torch', 'torchvision')) + '\n')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--constraint', str(constraints),
                    'ultralytics==8.4.182'], check=True, env=ENV)
    assert metadata.version('ultralytics') == '8.4.182'
    GUARD = WORK / 'loss-guard.json'
    if not GUARD.exists():
        subprocess.run([sys.executable, '-B', str(SOURCE / 'scripts/check_partial_classification_r51.py'),
                        '--root', str(SOURCE), '--inputs', str(INPUTS), '--dataset-archive', str(ARCHIVE),
                        '--output', str(GUARD)], check=True, env=ENV)
    if not (WORK / 'cpu-loader-preflight.json').exists():
        run('cpu-loader')
    run('verify-cpu')
    print('CPU native-equivalence, gradient and actual-loader gates passed.')
else:
    print('Verified completed result: CPU evidence is included in the return.')


## Run the matched pair and return one ZIP

The four-batch factor2 trial checks resource fit, finite gradients and actual updates; neither arm resumes from that trial. Each full arm starts from epoch173 and must consume exactly **1,080 batches / 4,260 image presentations / 80 optimizer updates**. Completed arms can be verified and reused; interrupted runs are preserved.

Evaluation uses each arm's **last.pt**, confidence **0.13**, the same native NMS and postclip filter, and the same four reference versions. The ZIP includes both final checkpoints, measured budgets, classification-controller observations, CPU guard evidence, predictions, paired missed-row changes, corner strata and source/label inventories. These repeated development scenes are not an independent test set.

Optional: set `BAOMAI_R51_BACKUP` to a writable persistent folder before running; each completed trial/arm is copied there with hash-verified markers. No cloud drive is mounted automatically.

In [ ]:
# 4. Four-batch trial, fresh matched arms, fixed evaluation, verified ZIP.
import shutil, zipfile
BACKUP = Path(os.environ['BAOMAI_R51_BACKUP']) if os.environ.get('BAOMAI_R51_BACKUP') else None
if not COMPLETED:
    options = ['--backup-dir', str(BACKUP)] if BACKUP else []
    run('run-pair', *options)
    SAVED_RETURN = BACKUP / RESULT.name if BACKUP else None
    if SAVED_RETURN and SAVED_RETURN.exists():
        run('restore-return', '--output', str(SAVED_RETURN))
    else:
        run('evaluate')
    run('package', '--output', str(RESULT))
    if SAVED_RETURN:
        if SAVED_RETURN.exists():
            assert hashlib.sha256(SAVED_RETURN.read_bytes()).digest() == hashlib.sha256(RESULT.read_bytes()).digest(), 'Different completed return preserved.'
        else:
            SAVED_RETURN.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(RESULT, SAVED_RETURN)
        run('verify-return', '--output', str(SAVED_RETURN))
else:
    run('verify-return', '--output', str(RESULT))
with zipfile.ZipFile(RESULT) as archive:
    summary = json.loads(archive.read('evaluation/summary.json'))
for name in ('frozen666', 'historical663', 'r38', 'r50'):
    print('\nReference:', name)
    for role in ('reference', 'candidate'):
        report = summary['references'][name]['postclip'][role]
        print(role, {k: report['detection'][k] for k in ('tp', 'fn', 'fp', 'duplicate_fp', 'other_fp')},
              'two-corner5px:', report['corners_by_locatable_count']['2']['within5'])
print('Return:', RESULT, '| SHA256:', hashlib.sha256(RESULT.read_bytes()).hexdigest())
try:
    from google.colab import files
    files.download(str(RESULT))
except ImportError:
    print('Download the ZIP from the path above.')
